# SIREN A3 — Inference Notebook
## Laiba Noor (25I-8035) | Group ID09 | FAST-NUCES
### Improvement IV + Bonus: Helmholtz k-Ablation · Wave IC Study · Brain MRI Super-Resolution

This notebook loads **trained checkpoints** and runs inference only — no retraining needed.

| Section | What it does |
|---|---|
| **1** | Setup & imports |
| **2** | Helmholtz Baseline inference (A2) |
| **3** | Wave Equation Baseline inference (A2) |
| **4** | Helmholtz k-Ablation inference (A3 Improvement IV-A) |
| **5** | Wave IC Variation inference (A3 Improvement IV-B) |
| **6** | Brain MRI Super-Resolution inference (A3 Bonus) |
| **7** | Summary table |


---
## Section 1 — Setup

In [ ]:
# Mount Google Drive (checkpoints saved here)
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/SIREN_A3_Laiba'
CKPT_ROOT  = '/content/drive/MyDrive/SIREN_A3_Laiba'  # adjust if different
print('Drive mounted.')

In [ ]:
# Clone SIREN repo and apply fixes
if not os.path.exists('/content/siren'):
    !git clone https://github.com/vsitzmann/siren.git /content/siren -q
    !sed -i 's/NeuralProcessImplicit2DHypernetBVP/NeuralProcessImplicit2DHypernet/g' /content/siren/utils.py
    !sed -i 's/from skimage.measure import compare_psnr, compare_ssim/from skimage.metrics import peak_signal_noise_ratio as compare_psnr, structural_similarity as compare_ssim/g' /content/siren/utils.py
    print('Repo cloned and fixed.')
else:
    print('Repo already exists.')

In [ ]:
!pip install configargparse cmapy scikit-video scikit-image -q

import sys, time
import numpy as np
import torch
import torch.nn as nn
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio as calc_psnr
from skimage.metrics import structural_similarity as calc_ssim
from skimage.transform import resize as sk_resize

sys.path.insert(0, '/content/siren')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# ── Model definitions (same as training notebook) ───────────────────────────
class SineLayer(nn.Module):
    def __init__(self, in_f, out_f, is_first=False, omega_0=30.):
        super().__init__()
        self.omega_0 = omega_0
        self.linear  = nn.Linear(in_f, out_f)
        with torch.no_grad():
            if is_first:
                self.linear.weight.uniform_(-1/in_f, 1/in_f)
            else:
                self.linear.weight.uniform_(-np.sqrt(6/in_f)/omega_0, np.sqrt(6/in_f)/omega_0)
    def forward(self, x):
        return torch.sin(self.omega_0 * self.linear(x))

class Siren(nn.Module):
    def __init__(self, in_f=2, hid=256, layers=4, out_f=1, omega_0=30.):
        super().__init__()
        self.net = nn.Sequential(
            SineLayer(in_f, hid, is_first=True, omega_0=omega_0),
            *[SineLayer(hid, hid, omega_0=omega_0) for _ in range(layers-1)],
            nn.Linear(hid, out_f)
        )
        with torch.no_grad():
            self.net[-1].weight.uniform_(-np.sqrt(6/hid)/omega_0, np.sqrt(6/hid)/omega_0)
    def forward(self, x): return self.net(x)

class ReLUMLP(nn.Module):
    def __init__(self, in_f=2, hid=256, layers=4, out_f=1):
        super().__init__()
        ls = [nn.Linear(in_f, hid), nn.ReLU()]
        for _ in range(layers-1): ls += [nn.Linear(hid,hid), nn.ReLU()]
        ls.append(nn.Linear(hid, out_f))
        self.net = nn.Sequential(*ls)
    def forward(self, x): return self.net(x)

print('Models defined.')

---
## Section 2 — Helmholtz Equation Baseline Inference (A2)
Loads `model_final.pth` from baseline training and visualises the wavefield.

In [ ]:
import dataio, modules

HELM_CKPT = './logs/helmholtz_repro/checkpoints/model_final.pth'
# ↑ Change path if your checkpoint is on Drive:
# HELM_CKPT = f'{CKPT_ROOT}/helmholtz_baseline/model_final.pth'

dataset = dataio.SingleHelmholtzSource(sidelength=512)
model_h = modules.SingleBVPNet(in_features=2, out_features=2,
                                type='sine', mode='mlp',
                                hidden_features=256, num_hidden_layers=3)
model_h.load_state_dict(torch.load(HELM_CKPT, map_location=device))
model_h.to(device).eval()
print(f'Loaded Helmholtz baseline: {HELM_CKPT}')

In [ ]:
with torch.no_grad():
    model_input, gt = dataset[0]
    model_input = {k: v.unsqueeze(0).to(device) for k, v in model_input.items()}
    out = model_h(model_input)

pred = out['model_out'].squeeze().cpu().numpy()
res = 512
pred_real = pred[:, 0].reshape(res, res)
pred_imag = pred[:, 1].reshape(res, res)
pred_amp  = np.sqrt(pred_real**2 + pred_imag**2)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, data, cmap, title in zip(axes,
        [pred_real, pred_imag, pred_amp],
        ['RdBu', 'RdBu', 'hot'],
        ['Real Part', 'Imaginary Part', 'Amplitude |u|']):
    im = ax.imshow(data, cmap=cmap, extent=[-1,1,-1,1], origin='lower')
    ax.set_title(f'Helmholtz Baseline — {title}', fontweight='bold')
    ax.set_xlabel('x'); ax.set_ylabel('y')
    plt.colorbar(im, ax=ax)
plt.suptitle('A2 Baseline: Helmholtz Wavefield (5000 steps)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/inference_helmholtz_baseline.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved inference_helmholtz_baseline.png')

---
## Section 3 — Wave Equation Baseline Inference (A2)
Loads `model_final.pth` from wave baseline and visualises the spacetime solution u(x,t).

In [ ]:
WAVE_CKPT = './logs/wave_repro/checkpoints/model_final.pth'
# WAVE_CKPT = f'{CKPT_ROOT}/wave_baseline/model_final.pth'

model_w = modules.SingleBVPNet(in_features=3, out_features=1,
                                type='sine', mode='mlp',
                                hidden_features=256, num_hidden_layers=3)
model_w.load_state_dict(torch.load(WAVE_CKPT, map_location=device))
model_w.to(device).eval()
print(f'Loaded Wave baseline: {WAVE_CKPT}')

In [ ]:
res = 128
x = torch.linspace(-1, 1, res)
t = torch.linspace(0, 1, res)
xx, tt = torch.meshgrid(x, t, indexing='ij')
coords = torch.stack([xx.flatten(), torch.zeros_like(xx.flatten()), tt.flatten()], dim=-1)

with torch.no_grad():
    pred_w = model_w({'coords': coords.unsqueeze(0).to(device)})['model_out'].squeeze().cpu().numpy()

pred_wave = pred_w.reshape(res, res)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
im0 = axes[0].imshow(pred_wave, cmap='RdBu', extent=[0,1,-1,1], origin='lower', aspect='auto')
axes[0].set_title('Wave Baseline — u(x,t)', fontweight='bold')
axes[0].set_xlabel('t'); axes[0].set_ylabel('x')
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(np.abs(pred_wave), cmap='hot', extent=[0,1,-1,1], origin='lower', aspect='auto')
axes[1].set_title('Wave Baseline — |u(x,t)| Amplitude', fontweight='bold')
axes[1].set_xlabel('t'); axes[1].set_ylabel('x')
plt.colorbar(im1, ax=axes[1])

plt.suptitle('A2 Baseline: Wave Equation Spacetime Solution (5000 steps)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/inference_wave_baseline.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved inference_wave_baseline.png')

---
## Section 4 — Helmholtz k-Ablation Inference (A3 Improvement IV-A)
Loads checkpoints for k ∈ {1, 5, 20, 50} and compares wavefields side by side.

In [ ]:
K_VALUES = [1, 5, 20, 50]
H_SIDE   = 150
COLORS   = ['#1f77b4','#ff7f0e','#2ca02c','#d62728']

# ── Build the same coordinate grid used in training ──────────────────────────
def build_helm_coords(side):
    lin = torch.linspace(-1, 1, side)
    x, y = torch.meshgrid(lin, lin, indexing='ij')
    return torch.stack([x.reshape(-1), y.reshape(-1)], dim=-1)

coords_h = build_helm_coords(H_SIDE)

# ── Load checkpoints ─────────────────────────────────────────────────────────
# Checkpoint folder structure expected:
#   DRIVE_ROOT/helmholtz_k1/latest_checkpoint.pt
#   DRIVE_ROOT/helmholtz_k5/latest_checkpoint.pt  etc.
helm_preds = {}
for k in K_VALUES:
    ckpt_path = f'{CKPT_ROOT}/helmholtz_k{k}/latest_checkpoint.pt'
    if not os.path.exists(ckpt_path):
        print(f'  [SKIP] k={k} — checkpoint not found: {ckpt_path}')
        continue
    model = Siren(in_f=2, hid=256, layers=4, out_f=2).to(device)
    ckpt  = torch.load(ckpt_path, map_location=device)
    # handle both raw state_dict and wrapped checkpoint
    state = ckpt.get('model_state_dict', ckpt) if isinstance(ckpt, dict) else ckpt
    model.load_state_dict(state)
    model.eval()
    with torch.no_grad():
        wf = model(coords_h.to(device)).cpu().numpy()
    helm_preds[k] = wf
    print(f'  Loaded k={k} from {ckpt_path}')

print(f'Loaded {len(helm_preds)}/{len(K_VALUES)} Helmholtz checkpoints.')

In [ ]:
if helm_preds:
    S = H_SIDE
    loaded_ks = [k for k in K_VALUES if k in helm_preds]
    fig, axes = plt.subplots(3, len(loaded_ks), figsize=(6*len(loaded_ks), 15))
    if len(loaded_ks) == 1: axes = axes.reshape(3,1)  # handle single column

    row_titles = ['Real part  u_r', 'Imaginary part  u_i', 'Amplitude  |u|']
    for col, k in enumerate(loaded_ks):
        wf = helm_preds[k]
        r  = wf[:,0].reshape(S,S)
        im = wf[:,1].reshape(S,S)
        amp = np.sqrt(r**2 + im**2)
        for row, (data, cmap) in enumerate(zip([r, im, amp], ['seismic','seismic','inferno'])):
            vmax = np.percentile(np.abs(data), 99) or 1
            kw = dict(vmin=-vmax, vmax=vmax) if row < 2 else {}
            ax = axes[row, col]
            i = ax.imshow(data, cmap=cmap, origin='lower', **kw)
            ax.set_title(f'k={k} | {row_titles[row]}', fontsize=11, fontweight='bold')
            ax.axis('off'); plt.colorbar(i, ax=ax, fraction=0.046)

    plt.suptitle('A3 Improvement IV-A: Helmholtz k-Ablation Wavefields', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(f'{DRIVE_ROOT}/inference_helmholtz_kablation.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved inference_helmholtz_kablation.png')
else:
    print('No Helmholtz A3 checkpoints found — run training notebook first.')

---
## Section 5 — Wave Equation IC Variation Inference (A3 Improvement IV-B)
Loads checkpoints for IC-1 (sin), IC-2 (Gaussian), IC-3 (Double-sin) and compares spacetime solutions.

In [ ]:
IC_TYPES  = ['default', 'gaussian', 'double_sin']
IC_LABELS = {'default':'IC-1: sin(πx)', 'gaussian':'IC-2: Gaussian', 'double_sin':'IC-3: Double-sin'}
IC_COLORS = ['#1f77b4','#2ca02c','#d62728']

# Expected checkpoint paths:
#   DRIVE_ROOT/wave_default/latest_checkpoint.pt  etc.
wave_preds = {}
for ic in IC_TYPES:
    ckpt_path = f'{CKPT_ROOT}/wave_{ic}/latest_checkpoint.pt'
    if not os.path.exists(ckpt_path):
        print(f'  [SKIP] {ic} — checkpoint not found: {ckpt_path}')
        continue
    model = Siren(in_f=3, hid=256, layers=3, out_f=1).to(device)
    ckpt  = torch.load(ckpt_path, map_location=device)
    state = ckpt.get('model_state_dict', ckpt) if isinstance(ckpt, dict) else ckpt
    model.load_state_dict(state)
    model.eval()
    # spacetime grid y=0 slice
    lin = torch.linspace(-1,1,80); tv = torch.linspace(0,1,80)
    gx, gt_ = torch.meshgrid(lin, tv, indexing='ij')
    vc = torch.stack([gx.reshape(-1), torch.zeros_like(gx).reshape(-1), gt_.reshape(-1)], dim=-1)
    with torch.no_grad():
        u_vis = model(vc.to(device)).cpu().numpy().reshape(80,80)
    wave_preds[ic] = u_vis
    print(f'  Loaded IC={ic}')

print(f'Loaded {len(wave_preds)}/{len(IC_TYPES)} Wave checkpoints.')

In [ ]:
if wave_preds:
    loaded_ics = [ic for ic in IC_TYPES if ic in wave_preds]
    fig, axes = plt.subplots(2, len(loaded_ics), figsize=(6*len(loaded_ics), 11))
    if len(loaded_ics) == 1: axes = axes.reshape(2,1)

    for col, ic in enumerate(loaded_ics):
        uv = wave_preds[ic]
        vm = np.percentile(np.abs(uv), 99) or 1
        im1 = axes[0,col].imshow(uv, cmap='RdBu', origin='lower',
                                  vmin=-vm, vmax=vm, aspect='auto', extent=[-1,1,0,1])
        axes[0,col].set_title(f'{IC_LABELS[ic]}\nu(x,t)', fontsize=11, fontweight='bold')
        axes[0,col].set_xlabel('x'); axes[0,col].set_ylabel('t')
        plt.colorbar(im1, ax=axes[0,col], fraction=0.046)

        im2 = axes[1,col].imshow(np.abs(uv), cmap='hot', origin='lower',
                                  aspect='auto', extent=[-1,1,0,1])
        axes[1,col].set_title(f'{IC_LABELS[ic]}\n|u(x,t)| Amplitude', fontsize=11, fontweight='bold')
        axes[1,col].set_xlabel('x'); axes[1,col].set_ylabel('t')
        plt.colorbar(im2, ax=axes[1,col], fraction=0.046)

    plt.suptitle('A3 Improvement IV-B: Wave Equation IC Variation — Spacetime Solutions',
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(f'{DRIVE_ROOT}/inference_wave_ic.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved inference_wave_ic.png')
else:
    print('No Wave A3 checkpoints found — run training notebook first.')

---
## Section 6 — Brain MRI Super-Resolution Inference (A3 Bonus)
Loads the 5 saved MRI slices and runs all 4 methods (Bicubic, ReLU, SIREN, SIREN+FFL+CAOS) for 2× and 4× SR.

In [ ]:
from skimage.transform import resize as sk_resize

TARGET_SZ  = 128
SCALES_MRI = [2, 4]
MRI_STEPS  = 400
MRI_LR     = 1e-4

# ── Load saved MRI slices ────────────────────────────────────────────────────
MRI_NPY_DIR = f'{DRIVE_ROOT}/mri_slices'
hr_slices = []
for i in range(5):
    p = f'{MRI_NPY_DIR}/slice_{i}.npy'
    if os.path.exists(p):
        hr_slices.append(np.load(p))
        print(f'  Loaded slice_{i}.npy  shape={hr_slices[-1].shape}')
    else:
        print(f'  [MISSING] {p}')

if len(hr_slices) == 0:
    print('\nNo MRI slices found — generating synthetic ones...')
    np.random.seed(42)
    while len(hr_slices) < 5:
        sl = np.zeros((TARGET_SZ, TARGET_SZ), np.float32)
        y, x = np.ogrid[-1:1:TARGET_SZ*1j, -1:1:TARGET_SZ*1j]
        brain = (x**2/0.7**2 + y**2/0.85**2) < 1.0
        grey  = (x**2/0.5**2 + y**2/0.60**2) < 1.0
        sl[brain] = 0.3 + 0.08*np.random.randn(TARGET_SZ,TARGET_SZ)[brain]
        sl[grey&brain] += 0.35 + 0.06*np.random.randn(TARGET_SZ,TARGET_SZ)[grey&brain]
        sl = np.clip(sl + 0.04*np.random.randn(TARGET_SZ,TARGET_SZ)*brain, 0, 1)
        hr_slices.append(sl)
    print(f'Generated {len(hr_slices)} synthetic slices.')

print(f'\nTotal slices ready: {len(hr_slices)}')

In [ ]:
# ── Helper: CAOS omega schedule ───────────────────────────────────────────────
def caos_omega(step, total, lo=10., hi=60.):
    return lo + (hi-lo)*(1-np.cos(np.pi*step/total))/2

def make_grid(sz):
    lin = torch.linspace(-1,1,sz)
    gy, gx = torch.meshgrid(lin, lin, indexing='ij')
    return torch.stack([gx.reshape(-1), gy.reshape(-1)], dim=-1)

def downsample(img, s):
    return sk_resize(img, (img.shape[0]//s, img.shape[1]//s),
                     anti_aliasing=True, preserve_range=True)

def bicubic_up(lr, sz):
    return sk_resize(lr, (sz, sz), order=3, anti_aliasing=True, preserve_range=True)

def run_mri_inference(lr_img, hr_img, mtype='siren', use_ffl=False, use_caos=False):
    H  = lr_img.shape[0]; Hr = hr_img.shape[0]
    model = (Siren(in_f=2, hid=256, layers=5, out_f=1)
             if mtype=='siren' else ReLUMLP(in_f=2, hid=256, layers=5, out_f=1)).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=MRI_LR)
    clr   = make_grid(H).to(device)
    plr   = torch.from_numpy(lr_img.reshape(-1,1)).float().to(device)
    chr_  = make_grid(Hr).to(device)
    for step in range(MRI_STEPS):
        model.train()
        if use_caos and mtype=='siren' and step%40==0:
            w = caos_omega(step, MRI_STEPS)
            for m in model.net.modules():
                if isinstance(m, SineLayer): m.omega_0 = w
        pred = model(clr); l2 = ((pred-plr)**2).mean(); loss = l2
        if use_ffl:
            lam = 0.05*min(1., step/200)
            p2d = pred.reshape(H,H); g2d = plr.reshape(H,H)
            loss = l2 + lam*torch.mean(torch.abs(
                torch.abs(torch.fft.fft2(p2d)) - torch.abs(torch.fft.fft2(g2d))))
        opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        ph = model(chr_).cpu().numpy().reshape(Hr, Hr)
    ph = np.clip(ph, 0, 1)
    return {'psnr': calc_psnr(hr_img, ph, data_range=1.),
            'ssim': calc_ssim(hr_img, ph, data_range=1.),
            'pred': ph}

METHODS = [('bicubic','Bicubic',False,False),
           ('relu',   'ReLU MLP',False,False),
           ('siren',  'SIREN (A2)',False,False),
           ('siren_ffl_caos','SIREN+FFL+CAOS',True,True)]
print('Inference helpers ready.')

In [ ]:
# ── Run inference on slice 0 for quick visual check ──────────────────────────
hr = hr_slices[0]
mri_quick = {}

for scale in SCALES_MRI:
    lr = downsample(hr, scale)
    mri_quick[scale] = {}
    print(f'\n--- Scale {scale}× | LR={lr.shape} ---')
    for mkey, mname, ffl, caos in METHODS:
        if mkey == 'bicubic':
            bic = bicubic_up(lr, TARGET_SZ)
            res = {'psnr': calc_psnr(hr, bic, data_range=1.),
                   'ssim': calc_ssim(hr, bic, data_range=1.), 'pred': bic}
        else:
            mtype = 'relu' if mkey=='relu' else 'siren'
            res = run_mri_inference(lr, hr, mtype=mtype, use_ffl=ffl, use_caos=caos)
        mri_quick[scale][mkey] = res
        print(f'  {mname:<28} PSNR={res["psnr"]:6.2f}  SSIM={res["ssim"]:.4f}')

In [ ]:
# ── Visual comparison (slice 0) ───────────────────────────────────────────────
order  = ['bicubic','relu','siren','siren_ffl_caos']
titles = {'bicubic':'Bicubic','relu':'ReLU MLP',
          'siren':'SIREN (A2)','siren_ffl_caos':'SIREN+FFL\n+CAOS (A3)'}

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for row, scale in enumerate(SCALES_MRI):
    for col, mk in enumerate(order):
        ax = axes[row, col]
        d  = mri_quick[scale][mk]
        ax.imshow(d['pred'], cmap='gray', vmin=0, vmax=1)
        ax.set_title(f'{titles[mk]}\n{d["psnr"]:.2f} dB | SSIM {d["ssim"]:.3f}',
                     fontsize=9, fontweight='bold')
        ax.axis('off')
        if col == 0: ax.set_ylabel(f'{scale}× SR', fontsize=12, fontweight='bold')

plt.suptitle('A3 Bonus: MRI Super-Resolution Inference — Slice 0',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/inference_mri_visual.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved inference_mri_visual.png')

---
## Section 7 — Summary
Quick recap of all inference results from this notebook.

In [ ]:
print('='*70)
print('  SIREN A3 INFERENCE SUMMARY — Laiba Noor (25I-8035)')
print('='*70)

print('\n[A2 Baselines]')
print('  Helmholtz  → inference_helmholtz_baseline.png')
print('  Wave Eq.   → inference_wave_baseline.png')

print('\n[A3 Improvement IV-A — Helmholtz k-Ablation]')
print(f'  Checkpoints found: {list(helm_preds.keys())}')
print('  Output → inference_helmholtz_kablation.png')

print('\n[A3 Improvement IV-B — Wave IC Variation]')
print(f'  ICs found: {list(wave_preds.keys())}')
print('  Output → inference_wave_ic.png')

print('\n[A3 Bonus — MRI Super-Resolution (Slice 0)]')
for scale in SCALES_MRI:
    print(f'  {scale}× SR:')
    for mkey, mname, _, _ in METHODS:
        d = mri_quick[scale][mkey]
        print(f'    {mname:<28} PSNR={d["psnr"]:6.2f} dB  SSIM={d["ssim"]:.4f}')

print('\nAll outputs saved to:', DRIVE_ROOT)
print('='*70)